# Spatial CNV inference on Xenium

Imaging-based spatial transcriptomics (Xenium, CosMx, MERSCOPE) measures a few hundred to a few thousand genes in
tens to hundreds of thousands of cells, with a few hundred counts per cell. inferCNV-style profiles of single cells
are noisy at that depth, but the tissue coordinates add information, because neighboring tumor cells usually share
their copy number state. The tutorial uses

- {func}`infercnvpy.tl.detectability` to see which chromosome arms the panel and depth can resolve, and to plan a panel,
- {func}`infercnvpy.tl.smooth_cnv` to average each cell's CNV profile with its spatial neighbors,
- {func}`infercnvpy.tl.spatial_regions` to make per-cell labels (here tumor and normal calls) spatially coherent, so
  that a cell takes the label of its neighborhood when most of its neighbors disagree with it (more than 75% at the
  default `beta`),
- {func}`infercnvpy.pl.spatial_cnv` to plot the mean CNV of a chromosome or arm on the tissue,
- {func}`infercnvpy.tl.copy_number` to call aneuploid cells and clones from the raw counts.

The spatial tools read a spatial neighbor graph from `adata.obsp`, here from {func}`squidpy.gr.spatial_neighbors_knn`,
and any cells x cells graph works. To run the tutorial, install squidpy, gtfparse and the lazy readers of anndata with
the optional extras, and s3fs to read the data from S3: `pip install "infercnvpy[spatial,gtf,dask]" s3fs`.

The ideas are not new. InSituCNV {cite}`Jensen2025` smooths infercnvpy profiles over kNN neighbors on Xenium and CosMx,
SlideCNA {cite}`Zhang2025` pools spatial neighbors on Slide-seq, and STARCH {cite}`Elyanow2021` and CalicoST
{cite}`Ma2024` make neighboring cells prefer the same clone label. In a benchmark of CNV inference on spatial data,
{cite:t}`Han2026` found that aggregating cells on imaging platforms improved CNV recovery compared to native
single-cell resolution.

We use a Xenium Prime 5K section of a squamous cell lung carcinoma from {cite:t}`Bilous2026`, distributed under GEO
terms through the scverse spatial data catalogue. Squamous lung cancers typically lose 3p and gain 3q.

In [ ]:
import importlib.metadata
import time

import anndata as ad
import numpy as np
import pandas as pd
import pooch
import scanpy as sc
import squidpy as sq
import zarr

import infercnvpy as cnv

sc.set_figure_params(dpi=50, frameon=False, figsize=(4.5, 4.5))

## Load the data

The dataset is a SpatialData object on S3. We only need its table, which we open lazily with
{func}`anndata.experimental.read_lazy` and then load the counts, the cell metadata and the cell centroids into memory.
Images, transcripts and segmentation shapes are never downloaded.

In [ ]:
group = zarr.open_group(
    "s3://scverse-spatial-eu-central-1/.lamindb/cOjo8Fe6mVpB4Mu0.zarr/tables/table",
    mode="r",
    storage_options={"anon": True},
)
lazy = ad.experimental.read_lazy(group)
obs = lazy.obs.to_memory()[["cell_id", "total_counts", "cell_area"]]
obs.index = obs.pop("cell_id").astype(str).to_numpy()
adata = ad.AnnData(
    X=lazy.X.compute(),
    obs=obs,
    var=lazy.var.to_memory()[["gene_ids"]],
    obsm={"spatial": lazy.obsm["spatial"].compute()},
)
# To start from a Xenium Ranger output folder instead, read the gene counts and the cell centroids. gex_only=True
# leaves out the control probes and the unassigned codewords, which are not genes and must not enter the CNV profiles.
# adata = sc.read_10x_h5("outs/cell_feature_matrix.h5", gex_only=True)
# cells = pd.read_parquet("outs/cells.parquet").set_index("cell_id")
# adata.obsm["spatial"] = cells.loc[adata.obs_names, ["x_centroid", "y_centroid"]].to_numpy()
adata

We drop cells with fewer than 50 counts and log-normalize to 10,000 counts per cell. The CNV values are log fold
changes, so the normalization target sets their scale: with the default target (the median of the remaining cells),
most values are `log1p` of 0 or 1 count, which compresses the CNV values.

In [ ]:
sc.pp.filter_cells(adata, min_counts=50)
print(f"median counts per cell after filtering: {np.median(adata.obs['n_counts']):.0f}")
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata

## Annotate reference cells

inferCNV compares every cell to normal reference cells. The dataset has no cell type labels, so we cluster the
expression data and annotate the clusters with marker gene scores. The Prime 5K panel leaves out many classic,
highly expressed markers, e.g. PTPRC, KRT5 and COL1A1, which are missing from the filtered lists printed below. We
use T, B and myeloid cell markers for immune cells, endothelial markers, fibroblast markers and squamous epithelial
markers, as far as they are in the panel. A cluster whose best score is below 0.1 has no clear marker support, and we
label it "unclear" and leave it out of the reference. Immune cells, endothelial cells and fibroblasts are the
reference, and the epithelial cells contain the tumor.

In [ ]:
markers = {
    "immune": [
        "PTPRC",
        "CD2",
        "CD3E",
        "CD4",
        "CD8A",
        "MS4A1",
        "CD79A",
        "MZB1",
        "CD68",
        "CD14",
        "CD163",
        "CSF1R",
        "MRC1",
    ],
    "endothelial": ["PECAM1", "CLDN5", "CDH5", "KDR", "PLVAP", "TEK"],
    "fibroblast": ["COL1A1", "PDGFRA", "PDGFRB", "FAP", "THY1", "COL5A1", "POSTN"],
    "epithelial": ["KRT5", "SOX2", "TP63", "DSG3", "EPCAM", "CDH1"],
}
markers = {k: [g for g in v if g in adata.var_names] for k, v in markers.items()}
print(markers)
for name, genes in markers.items():
    sc.tl.score_genes(adata, genes, score_name=name, random_state=0)

sc.pp.pca(adata, random_state=0)
sc.pp.neighbors(adata, random_state=0)
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2, random_state=0)
scores = adata.obs.groupby("leiden", observed=True)[list(markers)].mean()
# the type with the best marker score, or "unclear" when no score reaches 0.1
labels = scores.idxmax(axis=1).where(scores.max(axis=1) >= 0.1, "unclear")
adata.obs["cell_type"] = adata.obs["leiden"].map(labels).astype("category")
reference_types = ["immune", "endothelial", "fibroblast"]
scores.round(2).assign(cell_type=labels, n_cells=adata.obs["leiden"].value_counts())

In [ ]:
cnv.pl.spatial_cnv(adata, color="cell_type", size=1, title="cell type (marker scores)")

## Gene positions

The genomic positions of the genes come from the GENCODE v44 annotation, downloaded once with {mod}`pooch`. The
Xenium panel lists Ensembl gene ids, which we match to the GTF.

In [ ]:
gtf = pooch.retrieve(
    "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/gencode.v44.annotation.gtf.gz",
    known_hash="sha256:01f817afed65feee863361b4baf30a95e722ee5c5d508ff77b04106ef7ba20d3",
    fname="gencode.v44.annotation.gtf.gz",
    path=pooch.os_cache("infercnvpy"),
)
cnv.io.genomic_position_from_gtf(gtf, adata, gtf_gene_id="gene_id", adata_gene_id="gene_ids")
print(f"{adata.var['chromosome'].notna().sum()} of {adata.n_vars} panel genes have a GENCODE position")
genes_per_chromosome = adata.var["chromosome"].value_counts()
genes_per_chromosome[genes_per_chromosome.index.str.fullmatch(r"chr[0-9]+")].agg(["min", "max"])

## Which arms can this panel see?

Before we infer CNVs, {func}`infercnvpy.tl.detectability` estimates how many cells of one type that share the same
change a test needs to detect it. A chromosome arm is the part of a chromosome on one side of its centromere, the
short arm p or the long arm q. The function looks at the gain of one copy (1.5x, three copies instead of two) and the
loss of one copy (0.5x) of a whole arm. A test detects a change when it finds it with a chance of 80%, at a rate of
false calls of 1 in 1,000 per arm. The function doesn't find CNVs in your cells, and a short answer (few cells)
means that a missing call is informative, not that a call is real.

The function uses the raw counts, the depth and the expression of each cell type, and it measures the noise on the
reference cells. We give the cell types as `groupby`, so that each type gets its own row. The reference types get rows
too, which show what a change in those cells would need.

In [ ]:
start = time.perf_counter()
cnv.tl.detectability(
    adata, layer="counts", groupby="cell_type", reference_key="cell_type", reference_cat=reference_types
)
print(f"tl.detectability: {time.perf_counter() - start:.0f} s")
print("adata.uns['detectability']:", list(adata.uns["detectability"]), "| adata.obs: 'detectability_information'")
cnv.pl.detectability(adata)

The heatmap shows the cells needed for a gain, per cell type (rows) and arm (columns), on a log scale. Lighter cells
need fewer cells and are easier to detect. Pass `metric="cells_loss"` for losses. Arms with fewer than 5 panel genes
are hatched, and they are dark, because few genes carry little information. The p arms of chromosomes 13, 14, 15 and
22 hold no genes and have no column. chrY is left out by default, and chrX is included, which has one copy in cells of
a male donor.

The per-cell information (`obs["detectability_information"]`, summed over arms) shows where on the section the cells
carry the most information about copy number, mostly in the cells with the most counts.

In [ ]:
cnv.pl.spatial_cnv(adata, color="detectability_information", size=1, title="information per cell")

In [ ]:
table = adata.uns["detectability"]["table"]
arms = table[table["group"] == "epithelial"].assign(arm=lambda t: t["arm"].astype(str)).set_index("arm")
blind = arms.index[(arms["n_genes"] < 5) | ~np.isfinite(arms["cells_gain"])]
print(f"arms with fewer than 5 genes or no information: {', '.join(blind)}")
print(f"null_var_source: {', '.join(table['null_var_source'].unique())}")
columns = ["n_genes", "null_var", "cells_gain", "cells_gain_ideal", "cells_loss", "cells_loss_ideal"]
summary = arms.loc[["chr3p", "chr3q"], columns].astype(float)
summary.loc["median over arms"] = arms[columns].median()
summary.round(1)

For the epithelial cells, the median arm needs about 7 cells to detect a gain and about 5 cells to detect a loss
(table above). A few arms need more, e.g. 9p and 18p, and fibroblasts and immune cells need about twice as many cells
as epithelial cells on most arms (heatmap).

`null_var` says how much more an arm's signal varies in normal reference cells than counting noise alone predicts,
e.g. when a cell program moves the genes of an arm together. The achievable columns (`cells_gain`, `cells_loss`)
include it, and the ideal columns (`_ideal`) count only the noise of counting. Here `null_var` is about 1.1 for the
median arm, so the two columns are close. Plan with the achievable column. In a calibration on panel cuts of an Atera
breast sample, an ideal test, which knows the normal expression of every cell type, needed about as many cells as
the achievable column predicts.

The achievable column is still a lower bound for real methods. {func}`infercnvpy.tl.infercnv` estimates the normal
expression from the data, and in the same calibration it needed about 2 to 6 times as many cells on a 5K panel, and
on panels of 1,000 genes or fewer it missed about a third of the arm events (see the calibration table of
{func}`infercnvpy.tl.detectability`). The numbers also assume that expression follows the DNA copies one to one,
that the tumor is near diploid, and that all tested cells carry the change. For a near-tetraploid tumor, pass
`gain=1.25` and `loss=0.75`, and if only a share f of the tested cells carry the change, multiply the cells needed by
1/f². The function covers whole arms only, so focal events such as an amplification of SOX2 or MYC or a deletion of
CDKN2A are not covered.

The reference has no normal epithelium, so the epithelial row borrows its null variance from the immune and stromal
cells. It describes the counting noise of epithelial cells, and not how well tumor and normal epithelium separate.

In [ ]:
# What a smoothed neighborhood of 11 cells (a cell and its 10 neighbors, see below) can show on each arm
neighborhood = cnv.tl.detectability(
    adata,
    layer="counts",
    groupby="cell_type",
    reference_key="cell_type",
    reference_cat=reference_types,
    at_n_cells=11,
    inplace=False,
)
epithelial_11 = neighborhood[neighborhood["group"] == "epithelial"].assign(arm=lambda t: t["arm"].astype(str))
epithelial_11 = epithelial_11.set_index("arm")[["gain_ratio", "loss_ratio"]]
epithelial_11.loc["median over arms"] = epithelial_11.median()
epithelial_11.loc[["chr3p", "chr3q", "median over arms"]].round(2)

With `at_n_cells=11`, the table gives the smallest gain and the mildest loss that 11 epithelial cells show, the size
of a smoothed neighborhood below. For the median arm, an ideal test on 11 cells sees a gain to about 1.4x and a loss
to about 0.65x, so one copy (1.5x or 0.5x) is within reach of a neighborhood, but only just, and `tl.infercnv` needs
more cells than the ideal test.

## Planning a panel

The same function helps to choose a panel and a depth before an experiment.

1. Take raw counts with gene positions from a similar tissue, e.g. a droplet atlas or an earlier section.
2. Pass the genes of each candidate panel as `genes`, and the expected median counts per cell as `depth`. Use the
   counts per cell that the vendor gives for the panel and tissue.
3. Check that the arms you care about need fewer cells than you expect per clone or per spatial neighborhood, with
   room for the factor that `tl.infercnv` needs on top (see above).

Here we use this section as the expression data and compare its full panel with a random subset of 400 of its genes,
at 100 and 300 counts per cell. The random subset is only a stand-in for a smaller panel, since a real targeted panel
picks its genes by cell type and not at random.

In [ ]:
rng = np.random.default_rng(0)
panels = {"5K panel": list(adata.var_names), "400 random genes": list(rng.choice(adata.var_names, 400, replace=False))}
plans = {
    name: cnv.tl.detectability(
        adata,
        genes=genes,
        layer="counts",
        groupby="cell_type",
        reference_key="cell_type",
        reference_cat=reference_types,
        depth=[100, 300],
        inplace=False,
    )
    for name, genes in panels.items()
}
epithelial_plans = {name: plan[plan["group"] == "epithelial"] for name, plan in plans.items()}
cnv.pl.detectability(epithelial_plans)
pd.concat(
    {
        name: plan.groupby("depth").agg(
            median_cells_gain=("cells_gain", "median"), null_var_source=("null_var_source", "first")
        )
        for name, plan in epithelial_plans.items()
    }
).round(1)

With the 5K panel, the median arm of the epithelial cells needs about 15 cells at 100 counts per cell and about 6
cells at 300 counts. With the 400 random genes, it needs about 35 and 22 cells, many arms have fewer than 5 genes
(hatched), and some arms have no information at all (gray). At the same counts per cell, the smaller panel puts more
counts on each gene, and the extra variance of each gene's counts limits how much those extra counts help.

The 5K panel at 300 counts and the 400 genes at both depths lie above the counts that these genes have in this
section, so `tl.detectability` scales the expected counts up instead of thinning them (`null_var_source` is
"extrapolated"). The extrapolation was checked up to about 2 times the observed depth, so these rows are less certain
than the measured ones, and expression data at least as deep as the target is better. To turn cells needed into a
tissue area, divide by the density of the cell type in a comparable section.

## Infer CNVs and smooth them over the tissue

{func}`infercnvpy.tl.infercnv` runs as for single-cell data. The panel has 58 to 536 genes per autosome (printed above),
so we use running windows of 51 genes, every 5 genes, instead of the defaults
(100 and 10). We turn off the noise filter (`dynamic_threshold=None`), because it sets small values to 0 and
{func}`infercnvpy.tl.smooth_cnv` would then average those zeros. Then we build a spatial graph of the 10 nearest
neighbors of each cell with squidpy and average each cell's CNV profile with its neighbors. By default, the cell
itself counts as one more neighbor.

In [ ]:
start = time.perf_counter()
cnv.tl.infercnv(
    adata,
    reference_key="cell_type",
    reference_cat=reference_types,
    window_size=51,
    step=5,
    dynamic_threshold=None,
)
print(f"tl.infercnv: {time.perf_counter() - start:.0f} s")
sq.gr.spatial_neighbors_knn(adata, n_neighs=10)
cnv.tl.smooth_cnv(adata)
adata.obsm["X_cnv"].shape, adata.obsm["X_cnv_smoothed"].shape

In [ ]:
heatmap = {"groupby": "cell_type", "figsize": (12, 4), "vmin": -0.15, "vmax": 0.15}
cnv.pl.chromosome_heatmap(adata, **heatmap)
cnv.pl.chromosome_heatmap(adata, use_rep="cnv_smoothed", **heatmap)

Smoothing removes much of the noise of single cells, and the epithelial cells show a loss of 3p and a gain of 3q.
On the tissue, we plot the mean CNV of each arm of chromosome 3 (the windows whose midpoint lies in the arm, using the
window positions that `tl.infercnv` stores; the centromere is at about 91 Mb in GRCh38). The loss of 3p marks epithelial
regions of the section. The 3q map is red across most of the section, including regions of fibroblasts and immune
cells, so 3q is less specific here. The detectability table above predicted that both arms are easy to see on this
panel.

Part of the 3q signal of the epithelial cells may be lineage expression and not copy number. Squamous markers such as
SOX2 and TP63 sit on 3q, and the reference has no normal epithelium, so `tl.infercnv` compares squamous cells with
immune and stromal cells.

In [ ]:
arms = ["chr3:0-90_000_000", "chr3:93_000_000-199_000_000"]
cnv.pl.spatial_cnv(adata, region=arms, size=1, vmin=-0.06, vmax=0.06, title=["3p", "3q"])
cnv.pl.spatial_cnv(
    adata, region=arms, use_rep="cnv_smoothed", size=1, vmin=-0.06, vmax=0.06, title=["3p, smoothed", "3q, smoothed"]
)

## Tumor calls

We score each cell by the mean absolute value of its smoothed CNV profile and call it tumor when the score is above
the 95th percentile of the scores of the reference cells, so the score and the threshold are both per cell. By
construction, 5% of the reference cells are called tumor, and normal cells of types outside the reference can exceed
the threshold as well.

In [ ]:
reference = adata.obs["cell_type"].isin(reference_types).to_numpy()
cell_score = np.abs(adata.obsm["X_cnv_smoothed"]).mean(axis=1)
threshold = np.quantile(cell_score[reference], 0.95)
adata.obs["cnv_call"] = pd.Categorical(np.where(cell_score > threshold, "tumour", "normal"))
calls = pd.crosstab(adata.obs["cell_type"], adata.obs["cnv_call"])
print(f"threshold: {threshold:.4f}")
print(
    f"share of tumour calls that are not epithelial: {1 - calls.loc['epithelial', 'tumour'] / calls['tumour'].sum():.1%}"
)
calls

Most epithelial cells are called tumor. The fibroblasts, immune and endothelial cells among the tumor calls (share
printed above) are the 5% of reference cells above the threshold. The tumor calls show the loss of 3p, the gain of
3q and losses on chromosomes 4, 5 and 9.

In [ ]:
cnv.pl.chromosome_heatmap(adata, groupby="cnv_call", use_rep="cnv_smoothed", figsize=(12, 3), vmin=-0.15, vmax=0.15)

## Spatially coherent regions

{func}`infercnvpy.tl.spatial_regions` makes labels spatially coherent: a cell takes the label of its neighborhood
when enough neighbors disagree with it (for two labels and the default `beta=2`, more than 75% of them).
The calls from the smoothed profiles are already spatially coherent, and few cells change.

In [ ]:
cnv.tl.spatial_regions(adata, key="cnv_call")
print(f"{(adata.obs['cnv_call'] != adata.obs['cnv_call_spatial']).sum()} of {adata.n_obs} cells changed their label")
pd.crosstab(adata.obs["cnv_call"], adata.obs["cnv_call_spatial"])

The prior matters more for noisy labels. Calls of single cells from the unsmoothed CNV profiles, with the same rule,
are scattered over the section (left in the figure below). With `beta=4` (a cell changes its label when more than
62.5% of its neighbors disagree), `tl.spatial_regions` removes about half of the tumor calls (8,978 to 4,842 in the
table below), including 87% of the calls of non-epithelial cells (middle). It doesn't recover tumor cells that the
unsmoothed profiles miss, so the regularized calls from the smoothed profiles (right) include more than twice as many
epithelial cells.

In [ ]:
raw_score = np.asarray(abs(adata.obsm["X_cnv"]).mean(axis=1)).ravel()
raw_threshold = np.quantile(raw_score[reference], 0.95)
adata.obs["cnv_call_raw"] = pd.Categorical(np.where(raw_score > raw_threshold, "tumour", "normal"))
cnv.tl.spatial_regions(adata, key="cnv_call_raw", beta=4)
epithelial = np.where(adata.obs["cell_type"] == "epithelial", "epithelial", "other")
keys = ["cnv_call_raw", "cnv_call_raw_spatial", "cnv_call_spatial"]
pd.DataFrame({key: (adata.obs[key] == "tumour").groupby(epithelial).sum() for key in keys}).T.rename_axis(
    "tumour calls"
)

In [ ]:
cnv.pl.spatial_cnv(adata, color=["cnv_call_raw", "cnv_call_raw_spatial", "cnv_call_spatial"], size=1)

## Clones with tl.copy_number

{func}`infercnvpy.tl.copy_number` calls copy number from the raw counts with a count model. It returns aneuploid and
diploid labels, clones and segments (see {doc}`the API overview <../api>`). A clone is a group of cells that share the
same copy number profile. We give it the same reference cells and smooth its results over the spatial graph from
above, with `connectivity_key`. The graph must be built on the same cells as `adata`, in the same order, and with
`n_hops=1` it mixes each cell's result with its direct neighbors, which can spread calls into adjacent normal cells
at the tumor border.

Xenium cells have fewer detected genes than droplet cells, so we lower `min_genes` from 200 to 100, and cells below it
are labelled "not.defined". The distribution of detected genes per cell below shows how many cells that removes.

In [ ]:
detected = pd.Series(np.asarray((adata.layers["counts"] > 0).sum(axis=1)).ravel())
print(detected.describe(percentiles=[0.1, 0.25, 0.5]).round(0).to_string())
print(f"cells with fewer than 100 detected genes: {(detected < 100).mean():.1%}")

In [ ]:
start = time.perf_counter()
cnv.tl.copy_number(
    adata,
    layer="counts",
    reference_key="cell_type",
    reference_cat=reference_types,
    min_genes=100,
    connectivity_key="spatial_connectivities",
)
print(f"tl.copy_number: {time.perf_counter() - start:.0f} s")
# the numbers of the small-panel check, after the filters of min_genes and min_detection
used = adata.uns["copy_number"]["genes"].index
called = (adata.obs["copy_number"] != "not.defined").to_numpy()
median_counts = np.median(np.asarray(adata[called, used].layers["counts"].sum(axis=1)).ravel())
print(f"genes passing the filters: {len(used)} (check: at least 1,000)")
print(f"median counts per cell in them: {median_counts:.0f} (check: at least 150)")
print("adata.uns['copy_number']:", list(adata.uns["copy_number"]))
print("adata.obs:", [k for k in adata.obs if k.startswith("copy_number")])
print(adata.obs["copy_number"].value_counts())
adata.uns["copy_number"]["clones"]

The panel passes the check of `tl.copy_number` for small panels (at least 1,000 genes after its filters and a median
of 150 counts per cell in them, printed above), so it calls the cells instead of labelling them "undetermined".

In [ ]:
# one palette for the labels and the clones: aneuploid and T1 red, diploid and N0 blue, not.defined gray
palette = {"aneuploid": "#d62728", "T1": "#d62728", "diploid": "#1f77b4", "N0": "#1f77b4", "not.defined": "#c7c7c7"}
for key in ["copy_number", "copy_number_clone"]:
    adata.obs[key] = adata.obs[key].cat.remove_unused_categories()
    adata.uns[f"{key}_colors"] = [palette.get(c, "#9467bd") for c in adata.obs[key].cat.categories]
cnv.pl.spatial_cnv(adata, color=["copy_number", "copy_number_clone", "copy_number_spillover"], size=1)
cnv.pl.chromosome_heatmap(adata, groupby="copy_number_clone", use_rep="copy_number", figsize=(12, 3))

`tl.copy_number` finds one aneuploid clone, T1, and labels most other cells diploid (clone N0). Diploid means that no
arm-level change was detected, and not that the cell is normal. Cells with fewer than 100 detected genes are
"not.defined", which we treat as unknown. In the clone heatmap, T1 shows the loss of 3p and the gain of 3q, together
with gains of 5p and 8q and losses of 5q, 9, 13q and 18. We have no DNA measurement of the section to check the
profile, the weakest of these changes are 9p and 18p, and one clone in the model doesn't mean that the tumor is
monoclonal.

The spillover score is the share of a cell's counts that the pooled expression of the aneuploid cells explains best.
Read it only in the normal cell types, where high values point to segmentation merges with tumor cells, and also to
other causes, such as macrophages that take up tumor material or transcripts that diffuse between cells. The table
below compares the labels with the threshold calls from above, per cell type.

In [ ]:
cell_type = adata.obs["cell_type"]
pd.DataFrame(
    {
        # share of cells per type: threshold tumor calls of tl.infercnv, and the labels of tl.copy_number
        "threshold tumour calls": (adata.obs["cnv_call_spatial"] == "tumour").groupby(cell_type, observed=True).mean(),
        "copy_number aneuploid": (adata.obs["copy_number"] == "aneuploid").groupby(cell_type, observed=True).mean(),
        "copy_number not.defined": (adata.obs["copy_number"] == "not.defined").groupby(cell_type, observed=True).mean(),
        "median spillover": adata.obs["copy_number_spillover"].groupby(cell_type, observed=True).median(),
        "spillover above 0.5": (adata.obs["copy_number_spillover"] > 0.5).groupby(cell_type, observed=True).mean(),
    }
).round(3)

`tl.copy_number` labels fewer cells aneuploid than the threshold calls, in the epithelial cells and in the reference
types. The threshold rule calls 5% of the reference cells tumor by construction, while `tl.copy_number` fits its
calls without a fixed rate, so we prefer its labels where it calls the cells, and treat "not.defined" cells as
unknown. A few percent of the reference cells have a spillover above 0.5, so few of them look like merges with tumor
cells.

## Notes

- With 10 neighbors and the default weight of the cell itself, a cell's own profile is 1/11 of its smoothed profile,
  so calls from smoothed profiles are calls of neighborhoods. Smoothing mixes the profiles of tumor and normal cells
  at the tumor boundary and of immune cells that infiltrate the tumor, over about `n_hops` cells. Use the unsmoothed
  `X_cnv` for per-cell questions in mixed regions.
- The spatial graph defines the neighborhood. A kNN graph adapts to the cell density, and a radius graph
  ({func}`squidpy.gr.spatial_neighbors_radius`) keeps the physical scale fixed. `n_hops=2` in `tl.smooth_cnv`
  widens the neighborhood further.
- `tl.spatial_regions` changes labels only as long as a single change makes them more coherent, so the result stays
  close to the input labels and is not a new clustering. Larger `beta` gives smoother regions.
- For several sections in one AnnData, build the graph per section with `library_key` in
  {func}`squidpy.gr.spatial_neighbors_knn`, so that no cell gets neighbors from another section.
- On small panels (fewer than 1,000 usable genes, e.g. Xenium 480) or below a median of 150 counts per cell,
  `tl.copy_number` labels the cells "undetermined". Run `tl.detectability` to see which arms the panel can resolve,
  and then `tl.infercnv` with `tl.smooth_cnv`. The calls of `tl.copy_number(force_calls=True)` are not calibrated
  on such panels.

In [ ]:
# versions used to run this notebook
print("infercnvpy", cnv.__version__)
for package in ["anndata", "scanpy", "squidpy", "numpy", "pandas", "numba"]:
    print(package, importlib.metadata.version(package))